# Image-Plane Gaussian Fitting with `imfit`
## Dave Mehringer, April 2026

[Open in Colab](https://colab.research.google.com/github/casangi/astroviper/blob/main/docs/core_tutorials/image_analysis/imfit.ipynb)

This notebook demonstrates `imfit`, the astronomer-facing 2-D Gaussian fitter
for xradio image Datasets. It covers:

1. Building a synthetic xradio image with known Gaussian sources and a beam set
2. Running `imfit` to recover the source parameters
3. Inspecting sky coordinates, deconvolved sizes, and unresolved flagging
4. Verifying results against ground truth

## Install AstroVIPER

This cell only installs AstroVIPER when running in Google Colab.
Skip or ignore it when running locally.

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os
    os.system("pip install --upgrade astroviper")

from importlib.metadata import version
print(f"Running in Colab: {IN_COLAB}")
print(f"astroviper version: {version('astroviper')}")

## 1. Generate a Synthetic xradio Image Set

We build a test Dataset with:
- **1 time** step, **2 polarizations** (I, Q), **3 frequency** channels
- **120 x 100** pixels in (l, m)
- **2 Gaussian sources per plane**, with parameters that vary across planes
- Gaussian noise with peak S/N = 20 and min S/N = 5
- A per-plane **beam set** for deconvolution testing
- Pixel size chosen so that the smallest beam is Nyquist-sampled (pixel <= FWHM_min / 3)

In [ ]:
import numpy as np
import xarray as xr

from xradio.image import make_empty_sky_image
from astroviper.distributed_applications.model.component_models import make_gauss2d


def make_imfit_test_image(seed=42):
    """Build a synthetic xradio image Dataset for imfit tutorials.

    Uses ``make_empty_sky_image`` for the base Dataset and ``make_gauss2d``
    for source construction.  The l-axis is **decreasing** (positive →
    negative), so the pixel index for a source at l0 = k*cellsize is
    ``nl//2 - k``.  The m-axis is increasing, so m0 = k*cellsize maps to
    pixel ``nm//2 + k``.

    The Dataset has dimensions (time=1, frequency=3, polarization=2, l=120, m=100)
    with 2 Gaussian sources per plane whose parameters vary across planes,
    Gaussian noise scaled so peak S/N = 20 and min S/N = 5, a per-plane beam
    set, and sky coordinate grids.

    Returns
    -------
    xds : xr.Dataset
        Synthetic xradio image Dataset.
    truth : dict
        Ground-truth source parameters keyed by (pol_idx, freq_idx) plane,
        each containing a list of 2 component dicts.
    """
    rng = np.random.default_rng(seed)

    # --- Grid parameters ---
    nl, nm = 120, 100
    # Smallest beam minor FWHM = 4e-5 * 0.70 = 2.8e-5 rad => FWHM/3 = 9.33e-6
    cellsize = 9.0e-6  # radians per pixel (~1.86 arcsec), satisfies Nyquist

    # --- Outer dimensions ---
    n_time = 1
    n_freq = 3
    n_pol = 2
    time_vals = np.array([5.1e9])  # MJD-like
    freq_vals = np.array([1.0e9, 1.5e9, 2.0e9])
    pol_vals = np.array(["I", "Q"])
    phase_center = [0.5, -0.3]  # RA, Dec in radians

    # --- Build base xradio Dataset with proper coordinates and metadata ---
    xds = make_empty_sky_image(
        phase_center=phase_center,
        image_size=[nl, nm],
        cell_size=[cellsize, cellsize],
        frequency_coords=freq_vals,
        pol_coords=pol_vals,
        time_coords=time_vals,
        direction_reference="icrs",
        projection="SIN",
    )

    # l is decreasing, m is increasing — use coords as produced by xradio
    l_coord = xds.coords["l"]
    m_coord = xds.coords["m"]

    # --- Per-plane beam parameters (FWHM in radians, PA in radians) ---
    # Beams get slightly smaller at higher frequency (realistic)
    beam_params = {}
    for fi in range(n_freq):
        scale = 1.0 - 0.15 * fi  # 1.0, 0.85, 0.70
        for pi in range(n_pol):
            bmaj = 7.0e-5 * scale
            bmin = 4.0e-5 * scale
            bpa = np.deg2rad(30.0 + 10 * fi)  # 30, 40, 50 deg
            beam_params[(pi, fi)] = {"bmaj": bmaj, "bmin": bmin, "bpa": bpa}

    smallest_beam_fwhm = min(b["bmin"] for b in beam_params.values())
    assert cellsize <= smallest_beam_fwhm / 3, (
        f"Pixel size {cellsize} > FWHM_min/3 = {smallest_beam_fwhm/3}"
    )

    # --- Source parameters per plane ---
    # 2 components per plane, varying across (pol, freq)
    # S/N range: 5 to 20 (amplitude / noise_sigma)
    noise_sigma = 0.05

    amp_values = [1.0, 0.5, 0.8, 0.35, 0.7, 0.25, 0.9, 0.45, 0.6, 0.55, 0.75, 0.40]

    # Source positions scattered around the image center (in pixels from center)
    positions_pix = [
        (15, -12), (-20, 18),   # plane (0,0)
        (10, 20), (-15, -10),   # plane (0,1)
        (25, -5), (-10, 25),    # plane (0,2)
        (18, 10), (-22, -15),   # plane (1,0)
        (-8, 22), (20, -20),    # plane (1,1)
        (12, -18), (-18, 12),   # plane (1,2)
    ]

    # FWHM in pixels (converted to radians below), PA in radians
    fwhm_specs_pix = [
        (12, 8, 0.3), (9, 6, 0.8),
        (14, 7, 0.1), (8, 5, 1.0),
        (11, 9, 0.5), (10, 6, 0.7),
        (13, 7, 0.2), (9, 8, 0.9),
        (10, 7, 0.4), (15, 8, 0.6),
        (12, 6, 0.15), (8, 7, 1.1),
    ]

    img = np.zeros((n_time, n_freq, n_pol, nl, nm), dtype=float)
    truth = {}
    comp_idx = 0

    for pi in range(n_pol):
        for fi in range(n_freq):
            # Build this plane as a 2D DataArray; make_gauss2d accumulates components
            plane = xr.DataArray(
                np.zeros((nl, nm), dtype=float),
                dims=("l", "m"),
                coords={"l": l_coord, "m": m_coord},
            )
            plane_comps = []
            for ci in range(2):
                amp = amp_values[comp_idx]
                lpos_pix, mpos_pix = positions_pix[comp_idx]
                fwhm_maj_pix, fwhm_min_pix, pa = fwhm_specs_pix[comp_idx]

                l0 = lpos_pix * cellsize
                m0 = mpos_pix * cellsize
                fwhm_maj = fwhm_maj_pix * cellsize
                fwhm_min = fwhm_min_pix * cellsize

                plane = make_gauss2d(
                    plane,
                    a=fwhm_maj,
                    b=fwhm_min,
                    theta=pa,
                    x0=l0,
                    y0=m0,
                    peak=amp,
                    x_coord="l",
                    y_coord="m",
                    angle="pa",
                )

                plane_comps.append({
                    "amp": amp,
                    "l0": l0,
                    "m0": m0,
                    "fwhm_maj": fwhm_maj,
                    "fwhm_min": fwhm_min,
                    "pa": pa,
                    "snr": amp / noise_sigma,
                })
                comp_idx += 1

            img[0, fi, pi] = plane.values
            truth[(pi, fi)] = plane_comps

    # Add noise
    img += rng.normal(0, noise_sigma, img.shape)

    # --- Add SKY image data variable ---
    xds["SKY"] = xr.DataArray(
        img,
        dims=("time", "frequency", "polarization", "l", "m"),
        coords={
            "time": xds.coords["time"],
            "frequency": xds.coords["frequency"],
            "polarization": xds.coords["polarization"],
            "l": l_coord,
            "m": m_coord,
        },
    )

    # --- Beam set: (time, frequency, polarization, beam_params_label) ---
    beam_data = np.zeros((n_time, n_freq, n_pol, 3))
    for fi in range(n_freq):
        for pi in range(n_pol):
            bp = beam_params[(pi, fi)]
            beam_data[0, fi, pi, :] = [bp["bmaj"], bp["bmin"], bp["bpa"]]

    xds["BEAM_FIT_PARAMS_SKY"] = xr.DataArray(
        beam_data,
        dims=("time", "frequency", "polarization", "beam_params_label"),
        coords={
            "time": xds.coords["time"],
            "frequency": xds.coords["frequency"],
            "polarization": xds.coords["polarization"],
            "beam_params_label": xds.coords["beam_params_label"],
        },
        attrs={"units": "rad"},
    )

    # --- Sky coordinate grids (tangent-plane approximation) ---
    ra0, dec0 = phase_center
    cos_dec = np.cos(dec0)
    L, M = np.meshgrid(l_coord.values, m_coord.values, indexing="ij")
    RA = ra0 + L / cos_dec
    DEC = dec0 + M
    xds["right_ascension"] = xr.DataArray(
        RA, dims=("l", "m"), coords={"l": l_coord, "m": m_coord}
    )
    xds["declination"] = xr.DataArray(
        DEC, dims=("l", "m"), coords={"l": l_coord, "m": m_coord}
    )

    return xds, truth

In [ ]:
xds, truth = make_imfit_test_image()
print(f"\nBeam units: {xds['BEAM_FIT_PARAMS_SKY'].attrs['units']}")
print(f"Pixel size: {abs(float(xds.l[1] - xds.l[0])):.2e} rad "
      f"({abs(float(xds.l[1] - xds.l[0])) * 206265:.1f} arcsec)")
print(f"\nS/N range of injected sources:")
all_snr = [c["snr"] for comps in truth.values() for c in comps]
print(f"  min = {min(all_snr):.0f}, max = {max(all_snr):.0f}")
xds

### Inspect the ground-truth source parameters

Each of the 6 planes (2 pol x 3 freq) has 2 Gaussian components with different
amplitudes, positions, sizes, and orientations.

In [ ]:
import pandas as pd

pol_labels = ["I", "Q"]
freq_labels = [f"{f/1e9:.1f} GHz" for f in xds.frequency.values]

rows = []
for (pi, fi), comps in sorted(truth.items()):
    for i, c in enumerate(comps):
        rows.append({
            "pol": pol_labels[pi],
            "freq": freq_labels[fi],
            "comp": i,
            "amp": c["amp"],
            "S/N": int(c["snr"]),
            'l0 (")'  : round(c["l0"] * 206265, 1),
            'm0 (")'  : round(c["m0"] * 206265, 1),
            'FWHM_maj (")': round(c["fwhm_maj"] * 206265, 1),
            'FWHM_min (")': round(c["fwhm_min"] * 206265, 1),
            "PA (deg)": round(float(np.rad2deg(c["pa"])), 1),
        })

df = pd.DataFrame(rows).set_index(["pol", "freq", "comp"])
df

### Quick-look at one plane

In [ ]:
import matplotlib.pyplot as plt
from astroviper.utils.plotting import generate_plot

l_arcsec = xds.l.values * 206265
m_arcsec = xds.m.values * 206265

for fi in range(3):
    for pi in range(2):
        plane = xds["SKY"].isel(time=0, frequency=fi, polarization=pi)
        fig, ax = generate_plot(
            plane,
            show_world_axes=True,
            x_coords=l_arcsec,
            y_coords=m_arcsec,
            title=f"pol={pol_labels[pi]}, {freq_labels[fi]}",
            figsize=(5, 4),
        )
        ax.set_xlabel('l (arcsec)')
        ax.set_ylabel('m (arcsec)')
        plt.show()
        plt.close(fig)

## 2. Run `imfit`

We fit 2 Gaussian components per plane across all 6 planes simultaneously.
Although `imfit` optimizes in pixel space internally, manual initial guesses
may be specified in either pixel or world coordinates. World-frame guesses are
converted to pixel coordinates before the fit so the optimizer sees values in a
reasonable numerical range. Sky positions may be given in frame-aware angular
form, including sexagesimal strings. World-frame width guesses require square
pixels. For this demo we let the auto-seeder find the peaks.

In [ ]:
from astroviper.distributed_applications.image_analysis.imfit import imfit

ds = imfit(xds, n_components=2, mask_var=None, return_residual=True, return_model=True)
ds

## 3. Inspect Results

### Fit success and variance explained

In [ ]:
import pandas as pd

pol_labels_disp = ["I", "Q"]
freq_labels_disp = [f"{f/1e9:.1f} GHz" for f in xds.frequency.values]

# success = fitter converged (True for all planes here)
print("Fit convergence (success=True means optimizer converged):")
print(ds["success"].values.squeeze())

print("\nVariance explained per plane:")
print(ds["variance_explained"].values.squeeze().round(3))

print()
print("Unresolved components (True = source smaller than beam;")
print("  deconvolved sizes are NaN — see fwhm_upper_limit for beam-size upper bound):")
rows = []
for pi in range(2):
    for fi in range(3):
        unres = ds["is_unresolved"].isel(time=0, frequency=fi, polarization=pi).values
        rows.append({
            "pol": pol_labels_disp[pi],
            "freq": freq_labels_disp[fi],
            "comp 0 unresolved": bool(unres[0]),
            "comp 1 unresolved": bool(unres[1]),
        })
pd.DataFrame(rows).set_index(["pol", "freq"])

### Sky coordinates

`imfit` translates the fitted (l, m) centers to RA/Dec using the sky coordinate
grids on the input Dataset.

In [ ]:
# Show RA/Dec for the first plane (pol=I, freq=1.0 GHz)
sel = dict(time=0, frequency=0, polarization=0)
print("Component centers (pol=I, freq=1.0 GHz):")
for ci in range(2):
    ra = np.rad2deg(ds["right_ascension"].isel(**sel, component=ci).values) * 3600
    dec = np.rad2deg(ds["declination"].isel(**sel, component=ci).values) * 3600
    print(f"  Component {ci}: RA = {ra:.2f} arcsec, Dec = {dec:.2f} arcsec")

### Deconvolved source sizes

Since we provided a beam set, `imfit` deconvolves each fitted component from
its per-plane beam and flags unresolved sources.

In [ ]:
print("Deconvolved sizes and flags (all planes, arcsec):\n")
for pi in range(2):
    for fi in range(3):
        sel = dict(time=0, frequency=fi, polarization=pi)
        print(f"  pol={pol_labels[pi]}, freq={freq_labels[fi]}:")
        for ci in range(2):
            d_maj = ds["fwhm_major_deconv"].isel(**sel, component=ci).values * 206265
            d_min = ds["fwhm_minor_deconv"].isel(**sel, component=ci).values * 206265
            d_pa = np.rad2deg(ds["pa_deconv"].isel(**sel, component=ci).values)
            unres = ds["is_unresolved"].isel(**sel, component=ci).values
            if unres:
                ulim = ds["fwhm_upper_limit"].isel(**sel, component=ci).values * 206265
                print(f"    Comp {ci}: UNRESOLVED (upper limit = {ulim:.2f}\")")
            else:
                print(f"    Comp {ci}: FWHM_maj={d_maj:.2f}\", "
                      f"FWHM_min={d_min:.2f}\", PA={d_pa:.1f} deg")

### Output angle convention

`imfit` reports only PA (position angle, east of north) -- no math-convention
angles are included in the output.

In [ ]:
print(f"Theta convention: {ds.attrs['theta_convention']}")
print(f"PA definition: {ds.attrs['pa_definition']}")
print(f"\nAngle variables in output: {[v for v in ds.data_vars if 'pa' in v or 'theta' in v]}")
print(f"Math-angle variables: {[v for v in ds.data_vars if '_math' in v]}")

## 4. Compare Fitted vs Ground Truth

Check recovery of amplitude, position, and size for one plane.

In [ ]:
# Compare for pol=I, freq=1.0 GHz plane
sel = dict(time=0, frequency=0, polarization=0)
true_comps = truth[(0, 0)]

print("pol=I, freq=1.0 GHz plane comparison:\n")
print(f"{'':>8} {'True':>10} {'Fitted':>10} {'Error':>10} {'Fit_err':>10}")
print("-" * 55)

for ci in range(2):
    tc = true_comps[ci]
    amp_fit = ds["amplitude"].isel(**sel, component=ci).values
    amp_err = ds["amplitude_err"].isel(**sel, component=ci).values
    l0_fit = ds["x0_world"].isel(**sel, component=ci).values
    m0_fit = ds["y0_world"].isel(**sel, component=ci).values
    fmaj_fit = ds["fwhm_major_world"].isel(**sel, component=ci).values
    fmin_fit = ds["fwhm_minor_world"].isel(**sel, component=ci).values

    print(f"\nComponent {ci} (S/N={tc['snr']:.0f}):")
    print(f"{'amp':>8} {tc['amp']:10.4f} {float(amp_fit):10.4f} "
          f"{float(amp_fit - tc['amp']):10.4f} {float(amp_err):10.4f}")
    print(f"{'l0\"':>8} {tc['l0']*206265:10.2f} {float(l0_fit)*206265:10.2f} "
          f"{float(l0_fit - tc['l0'])*206265:10.2f}")
    print(f"{'m0\"':>8} {tc['m0']*206265:10.2f} {float(m0_fit)*206265:10.2f} "
          f"{float(m0_fit - tc['m0'])*206265:10.2f}")
    print(f"{'FWmaj\"':>8} {tc['fwhm_maj']*206265:10.2f} {float(fmaj_fit)*206265:10.2f} "
          f"{float(fmaj_fit - tc['fwhm_maj'])*206265:10.2f}")
    print(f"{'FWmin\"':>8} {tc['fwhm_min']*206265:10.2f} {float(fmin_fit)*206265:10.2f} "
          f"{float(fmin_fit - tc['fwhm_min'])*206265:10.2f}")

### Residual image

A good fit should leave only noise in the residual.

In [ ]:
sel = dict(time=0, frequency=0, polarization=0)
data_plane = xds["SKY"].isel(**sel)
model_plane = ds["model"].isel(**sel)
resid_plane = ds["residual"].isel(**sel)

l_arcsec = data_plane.l.values * 206265
m_arcsec = data_plane.m.values * 206265

for arr, title in zip(
    [data_plane, model_plane, resid_plane],
    ["Data", "Model", "Residual"],
):
    fig, ax = generate_plot(
        arr,
        show_world_axes=True,
        x_coords=l_arcsec,
        y_coords=m_arcsec,
        title=f"{title} — pol=I, freq=1.0 GHz",
        figsize=(5, 5),
    )
    ax.set_xlabel('l (arcsec)')
    ax.set_ylabel('m (arcsec)')
    plt.show()
    plt.close(fig)

print(f"Residual RMS: {float(resid_plane.values.std()):.4f} "
      f"(injected noise sigma: 0.05)")